# Planetary composites: AE versus CE, surface-referenced tilt
This extends `composite_breakdown` without changing it. Every selected day is translated to its fitted surface centre; the mean constituent centre at depth defines the new **surface-to-depth tilt vector**. Its length is the composite tilt distance. This definition also works for individual profiles; compositing estimates its population mean and can cancel opposing tilts. The length of the mean vector is **not** the mean length of individual vectors.

Geographic east/north components use the same ROMS grid-angle rotation as `seacofs_tilt_tools`. Bearings point **towards the deeper centre**, clockwise from north. They are reversed relative to the previous deep-to-shallow tilt convention. Existing `TiltDis` and `TiltDir` are never used to select or measure these tilts.

The main comparison includes all available fitted depths, including beyond 1,000 m. Independent depth support, whole-eddy bootstrap intervals, and a fixed-membership check accompany it. Intervals describe sampling uncertainty, not ESP fitting error or dependence between distinct eddies.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HERE = Path.cwd().resolve()
ANALYSIS = next((p for p in (HERE, *HERE.parents) if (p/'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subdirectory')
WORK = ANALYSIS/'esp_population_composites'
for p in (ANALYSIS, WORK):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
ESP_ROOT = Path("/home/z5297792/ESP_zonodo")
if str(ESP_ROOT) not in sys.path:
    sys.path.insert(0, str(ESP_ROOT))
import functions as esp
import seacofs_tilt_tools as tilt
pd.set_option('display.max_columns', 60)

# Reuse the dataset's directional-radius helper for the full composite refit.
DATASET_SRC = ANALYSIS.parent/'seacofs_eddy_dataset_modular'/'src'
if str(DATASET_SRC) not in sys.path:
    sys.path.insert(0, str(DATASET_SRC))
import composite_breakdown_tools as cbt

import planetary_composite_tools as pct


In [ ]:
DOMINANCE_FACTOR = 2.0
ELLIPSE_FRAC = 1.0
MIN_PLAN_DEPTH_M = 3000.0
USE_PV_CACHE = True
SPLIT_DEPTH_M = 1000.0
TARGET_DEPTHS_M = [200., 500., 1000., 1500., 2000.]
BOOTSTRAPS = 500
SEED = 731
MIN_DIRECTION_DISTANCE_KM = 0.0  # zero vectors always have undefined direction
MIN_PLOT_EDDIES = 2  # plotting threshold only; all estimates remain in tables
WIDTH_KM, RES_KM = 400., 10.
RUN_VELOCITY_COMPOSITES = True
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
surface, _ = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
surface = tilt.add_pv_gradient_terms(surface, grid, core_mean=True, frac=ELLIPSE_FRAC,
    surface_method='esp_gaussian', averaging='nonlinear', use_cache=USE_PV_CACHE)
vertical = tilt.load_vert(paths)
planetary = surface.loc[(surface.topo_plan_ratio <= -np.log(DOMINANCE_FACTOR)) &
    (surface.h >= MIN_PLAN_DEPTH_M) & surface.Cyc.isin(['AE','CE'])].copy()
profile_audit, centred_profiles = pct.prepare_profiles(planetary, vertical, SPLIT_DEPTH_M)
display(profile_audit.groupby(['Cyc','profile_status'], dropna=False).size().rename('eddy_days'))
display(profile_audit.groupby(['Cyc','extent_group'],dropna=False).agg(
    eddy_days=('Day','size'), eddies=('Eddy','nunique'), median_max_depth=('max_fit_depth_m','median')))


## Depth-extent groups and reconstruction
`shallow` means an eddy-day's deepest valid fit is ≤ the split depth; `deep` means > it. The classification is made **before** reconstruction or display-depth restrictions. A track can contribute days to both groups as its observed extent changes. These are observed fit extents, not certified physical eddy bottoms: fitting failures can truncate profiles.

The `all` group is the baseline and overlaps the other two groups. Days receive equal weight, matching the breakdown notebook. By default the actual ESP velocity composites are computed, and centre statistics use precisely the levels that contributed finite velocity. Turning off reconstruction allows faster centre-only exploration, using valid fitted centres instead; that mode is explicitly labelled below.


In [ ]:
a = np.arange(-WIDTH_KM/2, WIDTH_KM/2 + RES_KM/2, RES_KM)
X, Y = np.meshgrid(a, a)
results = {}
for cohort in ['all','shallow','deep']:
    for cyc in ['AE','CE']:
        selected = profile_audit.loc[profile_audit.Cyc.eq(cyc) & profile_audit.profile_status.eq('usable')].copy()
        if cohort != 'all': selected = selected.loc[selected.extent_group.eq(cohort)]
        if selected.empty:
            print(cohort,cyc,': no usable profiles'); continue
        if RUN_VELOCITY_COMPOSITES:
            u,v,depths,counts,support,audit,members = pct.composite_days(
                selected,vertical,X,Y,return_centres=True,esp=esp)
            field = dict(u=u,v=v,depths=depths,counts=counts,support=support,audit=audit)
        else:
            members = centred_profiles.merge(selected[['Eddy','Day']],on=['Eddy','Day'],validate='many_to_one')[['Eddy','Day','Depth','xc','yc']]
            field = {}
        members = pct.geographic_members(members, grid.angle)
        stats, boot = pct.summarise(members,BOOTSTRAPS,SEED)
        results[cohort,cyc] = dict(members=members,stats=stats,bootstrap=boot,**field)
        print(cohort,cyc,':',len(members),'contributing levels;',members.Eddy.nunique(),'eddies')
if not results: raise ValueError('No usable planetary profiles')
print('Mode:', 'finite ESP contributors' if RUN_VELOCITY_COMPOSITES else 'valid fitted centres only')
centre_stats = pd.concat([r['stats'].assign(cohort=g,Cyc=c) for (g,c),r in results.items()],ignore_index=True)
centre_members = pd.concat([r['members'].assign(cohort=g,Cyc=c) for (g,c),r in results.items()],ignore_index=True)
display(centre_stats)


## Composite tilt and depth support
Component bands are 95% pointwise whole-eddy bootstrap intervals; the covariance and variance columns quantify individual-day spread. The distance band bootstraps the **norm of each resampled mean vector**, preserving x–y covariance. Close to zero this nonnegative norm is biased upward, so use the component intervals to assess whether the mean displacement differs from zero. Direction is undefined for a zero vector.

`coherence = length(mean vector) / mean(length)` approaches zero when constituent tilts cancel and one when they point together. Sparse depths remain in the tables; plotting support is controlled above.


In [ ]:
COLORS = {'AE':'firebrick','CE':'royalblue'}
for cohort in ['all','shallow','deep']:
    fig,axs=plt.subplots(1,5,figsize=(17,5),sharey=True,constrained_layout=True)
    for cyc in ['AE','CE']:
        if (cohort,cyc) not in results: continue
        s=results[cohort,cyc]['stats'].copy()
        ok=s.n_eddies.ge(MIN_PLOT_EDDIES)
        for ax,col,lo,hi in [(axs[0],'mean_east','mean_east_ci_low','mean_east_ci_high'),
                             (axs[1],'mean_north','mean_north_ci_low','mean_north_ci_high'),
                             (axs[2],'distance_km','distance_ci_low','distance_ci_high')]:
            ax.plot(s[col].where(ok),s.Depth,color=COLORS[cyc],label=cyc)
            ax.fill_betweenx(s.Depth,s[lo].where(ok),s[hi].where(ok),color=COLORS[cyc],alpha=.18)
        axs[2].plot(s.mean_member_distance_km.where(ok),s.Depth,':',color=COLORS[cyc],label=f'{cyc} mean individual distance')
        axs[3].plot(s.coherence.where(ok),s.Depth,color=COLORS[cyc])
        axs[4].plot(s.n_eddies,s.Depth,color=COLORS[cyc],label=f'{cyc} eddies')
        axs[4].plot(s.n_eddy_days,s.Depth,':',color=COLORS[cyc],label=f'{cyc} days')
    for ax,label in zip(axs,['East displacement (km)','North displacement (km)','Distance (km)','Vector coherence','Contributors']):
        ax.set_xlabel(label); ax.axhline(SPLIT_DEPTH_M,color='.6',ls='--',lw=.8)
    axs[0].set_ylabel('Depth (m)'); axs[0].invert_yaxis()
    axs[0].legend();axs[2].legend(fontsize=7);axs[4].legend(fontsize=7)
    fig.suptitle(f'Planetary {cohort}: surface-to-depth mean constituent tilt')
    plt.show()


## Windroses at exact depths
Two distinct distributions are shown. **Constituent roses** show the distribution of surface-to-depth daily displacements; the arrow is the bearing of the mean vector of all contributors. **Bootstrap roses** show uncertainty in the mean constituent vector, not additional independent eddies. Near-zero means can have widely dispersed bearings.

All panels use fixed 22.5° sectors and percentages of retained vectors. Zero vectors are omitted. The optional minimum-distance threshold affects rose bars only; retained counts are shown. No nearest-depth substitution or vertical interpolation is used. All panels within a figure share a radial scale.


In [ ]:
ROSE_COHORT = 'all'  # also 'shallow' or 'deep'
MAG_BINS_KM = [0,10,20,30,40,np.inf]
for kind,title in [('members','Constituent eddy-day displacements'),('bootstrap','Bootstrap mean constituent displacements')]:
    fig,axs=plt.subplots(2,len(TARGET_DEPTHS_M),figsize=(3.3*len(TARGET_DEPTHS_M),7),
        subplot_kw={'projection':'polar'},squeeze=False,constrained_layout=True)
    for i,cyc in enumerate(['AE','CE']):
        r=results.get((ROSE_COHORT,cyc))
        for j,depth in enumerate(TARGET_DEPTHS_M):
            ax=axs[i,j]
            if r is None:
                ax.set_axis_off();continue
            d=r[kind].loc[r[kind].Depth.eq(depth)]
            n=pct.rose(ax,d,MAG_BINS_KM,MIN_DIRECTION_DISTANCE_KM,cmap='Reds' if cyc=='AE' else 'Blues')
            s=r['stats'].loc[r['stats'].Depth.eq(depth)]
            support='' if s.empty else f"; {int(s.n_eddies.iloc[0])} eddies"
            ax.set_title(f'{cyc}, {depth:g} m\nn={n}/{len(d)}{support}',fontsize=9)
    top=max(ax.get_ylim()[1] for ax in axs.flat)
    for i,cyc in enumerate(['AE','CE']):
        r=results.get((ROSE_COHORT,cyc))
        for j,depth in enumerate(TARGET_DEPTHS_M):
            ax=axs[i,j];ax.set_ylim(0,top)
            if r is None:continue
            s=r['stats'].loc[r['stats'].Depth.eq(depth)]
            if len(s) and np.isfinite(s.bearing_deg.iloc[0]):
                theta=np.deg2rad(s.bearing_deg.iloc[0])
                ax.annotate('',xy=(theta,.9*top),xytext=(theta,0),arrowprops=dict(arrowstyle='->',color='black',lw=1.8))
    handles,labels=[],[]
    for row,cyc in zip(axs,['AE','CE']):
        h,l=next((ax.get_legend_handles_labels() for ax in row if ax.get_legend_handles_labels()[0]),([],[]))
        handles.extend(h);labels.extend([f'{cyc}: {label}' for label in l])
    fig.legend(handles,labels,title='Displacement (km)',loc='lower center',ncol=5,bbox_to_anchor=(.5,-.15))
    fig.suptitle(f'{ROSE_COHORT}: {title}; radial units = percent')
    plt.show()


## Fixed-membership sensitivity
Keep only eddy-days with every listed exact depth (including the surface). This holds the same days fixed across the comparison depths, separating population turnover from the depth variation of their mean centres. Missing required depths produce an empty result rather than silently relaxing the condition. Change the depth list if support is insufficient. The filter does not establish physical vertical continuity between sampled depths.


In [ ]:
MATCH_DEPTHS_M = [0.,500.,1000.,1500.]
matched_results={}
for cyc in ['AE','CE']:
    r=results.get(('all',cyc))
    if r is None:continue
    m=pct.matched_members(r['members'], MATCH_DEPTHS_M)
    print(cyc,': matched',m[['Eddy','Day']].drop_duplicates().shape[0],'days,',m.Eddy.nunique(),'eddies')
    if len(m): matched_results[cyc]=pct.summarise(m.loc[m.Depth.isin(MATCH_DEPTHS_M)],BOOTSTRAPS,SEED)[0]
fig,axs=plt.subplots(1,3,figsize=(11,5),sharey=True,constrained_layout=True)
for cyc,s in matched_results.items():
    available=results['all',cyc]['stats'].loc[lambda x:x.Depth.isin(MATCH_DEPTHS_M)]
    for ax,col,lo,hi in zip(axs,['mean_east','mean_north','distance_km'],
        ['mean_east_ci_low','mean_north_ci_low','distance_ci_low'],['mean_east_ci_high','mean_north_ci_high','distance_ci_high']):
        ax.plot(available[col],available.Depth,'--',color=COLORS[cyc],label=f'{cyc} available')
        ax.plot(s[col],s.Depth,'o-',color=COLORS[cyc],label=f'{cyc} matched')
        ax.fill_betweenx(s.Depth,s[lo],s[hi],color=COLORS[cyc],alpha=.15)
for ax,label in zip(axs,['East displacement (km)','North displacement (km)','Mean-vector distance (km)']):ax.set_xlabel(label)
axs[0].invert_yaxis();axs[0].set_ylabel('Depth (m)');axs[0].legend();plt.show()


## AE–CE differences and environmental context
The table compares mean vectors at each shared exact depth. Independent whole-eddy resamples of AE and CE populations give pointwise confidence intervals for **AE minus CE**; polarity populations are assumed distinct. These are descriptive population comparisons, not controlled causal effects.

The binned panel below repeats the mean-vector estimator within surface Rossby-number ranges at one exact depth, and reports support and bootstrap uncertainty. The grouping uses the day's surface environment; it does not imply the same environment at depth.


In [ ]:
contrast_rows=[]
for cohort in ['all','shallow','deep']:
    if (cohort,'AE') not in results or (cohort,'CE') not in results:continue
    ae,ce=results[cohort,'AE'],results[cohort,'CE']
    # Independent random streams for the disjoint polarity populations.
    _,ce_boot=pct.summarise(ce['members'],BOOTSTRAPS,SEED+1)
    common=np.intersect1d(ae['stats'].Depth,ce['stats'].Depth)
    for z in common:
        sa=ae['stats'].set_index('Depth').loc[z];sc=ce['stats'].set_index('Depth').loc[z]
        ba=ae['bootstrap'].loc[lambda x:x.Depth.eq(z)].set_index('draw')
        bc=ce_boot.loc[lambda x:x.Depth.eq(z)].set_index('draw')
        for col,bootcol in [('mean_east','east_km'),('mean_north','north_km'),('distance_km','distance_km')]:
            delta=(ba[bootcol]-bc[bootcol]).dropna()
            lo,hi=np.quantile(delta,[.025,.975]) if len(delta) else (np.nan,np.nan)
            contrast_rows.append(dict(cohort=cohort,Depth=z,metric=col,AE_minus_CE=sa[col]-sc[col],ci_low=lo,ci_high=hi,
                                      AE_eddies=sa.n_eddies,CE_eddies=sc.n_eddies))
ae_ce_contrasts=pd.DataFrame(contrast_rows)
display(ae_ce_contrasts)

CONTEXT_DEPTH_M=500.
RO_EDGES=[0.,.1,.2,.3,.5,1.,np.inf]
context_rows=[]
if 'Ro' in planetary:
    for cyc in ['AE','CE']:
        if ('all',cyc) not in results:continue
        d=results['all',cyc]['members'].loc[lambda x:x.Depth.eq(CONTEXT_DEPTH_M)].merge(
            planetary[['Eddy','Day','Ro']],on=['Eddy','Day'],validate='many_to_one')
        d['Ro_bin']=pd.cut(d.Ro.abs(),RO_EDGES,right=False)
        for label,g in d.groupby('Ro_bin',observed=True):
            s,_=pct.summarise(g,BOOTSTRAPS,SEED)
            context_rows.append(s.assign(Cyc=cyc,Ro_median=g.Ro.abs().median(),Ro_bin=str(label)))
context_stats=pd.concat(context_rows,ignore_index=True) if context_rows else pd.DataFrame()
if len(context_stats):
    display(context_stats)
    fig,axs=plt.subplots(1,3,figsize=(12,4),constrained_layout=True)
    for cyc in ['AE','CE']:
        s=context_stats.loc[context_stats.Cyc.eq(cyc)&context_stats.n_eddies.ge(MIN_PLOT_EDDIES)]
        for ax,col,lo,hi in zip(axs,['mean_east','mean_north','distance_km'],
            ['mean_east_ci_low','mean_north_ci_low','distance_ci_low'],['mean_east_ci_high','mean_north_ci_high','distance_ci_high']):
            ax.plot(s.Ro_median,s[col],'o-',color=COLORS[cyc],label=cyc)
            ax.fill_between(s.Ro_median,s[lo],s[hi],color=COLORS[cyc],alpha=.18)
            ax.set(xlabel='Surface |Ro|',ylabel=f'{col} (km)');ax.legend()
    fig.suptitle(f'Planetary mean constituent tilt at {CONTEXT_DEPTH_M:g} m');plt.show()
else:print('No Rossby-number context at this exact depth')


## What do the velocity composites look like?
Compare AE and CE speed on one common colour scale, with the mean constituent centre marked in the original model-grid axes. These centres need not coincide with the velocity maximum or the centre obtained by refitting the composite. The inner/outer fit workflow remains available in `composite_breakdown`; this notebook's tilt estimator uses constituent centres.


In [ ]:
MAP_COHORT='all'
MAP_DEPTH_M=500.
fields=[]
for cyc in ['AE','CE']:
    r=results.get((MAP_COHORT,cyc))
    if r is not None and 'u' in r and MAP_DEPTH_M in r['depths']:
        k=int(np.flatnonzero(r['depths']==MAP_DEPTH_M)[0])
        fields.append((cyc,r,k,np.hypot(r['u'][...,k],r['v'][...,k])))
if fields:
    vmax=max(np.nanmax(f) for _,_,_,f in fields)
    fig,axs=plt.subplots(1,len(fields),figsize=(6*len(fields),5),squeeze=False,constrained_layout=True)
    for ax,(cyc,r,k,f) in zip(axs.flat,fields):
        im=ax.pcolormesh(X,Y,f,shading='auto',cmap='magma',vmin=0,vmax=vmax)
        m=r['members'].loc[lambda x:x.Depth.eq(MAP_DEPTH_M)]
        ax.plot(m.xc.mean(),m.yc.mean(),'co',label='Mean constituent centre')
        ax.plot(0,0,'w+',label='Surface reference');ax.legend(fontsize=8)
        ax.set(aspect='equal',xlabel='Model-grid x (km)',ylabel='Model-grid y (km)',title=f'{cyc}, {MAP_DEPTH_M:g} m; {len(m)} days')
    fig.colorbar(im,ax=list(axs.flat),label='Speed of mean ESP velocity (m/s)');plt.show()
else:print('No velocity composites at this exact depth; enable RUN_VELOCITY_COMPOSITES or choose an available depth')


## Optional export
`centre_members`, `centre_stats`, `ae_ce_contrasts`, `context_stats`, `profile_audit`, and `results` remain available for further exploration. Rows in `all` overlap the split groups: never pool the three cohorts. Saved tables include the estimator and cohort settings in a sidecar JSON.


In [ ]:
SAVE_TABLES=False
if SAVE_TABLES:
    out=Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/planetary_composite_tilt')
    out.mkdir(parents=True,exist_ok=True)
    for name in ['centre_members','centre_stats','ae_ce_contrasts','context_stats','profile_audit']:
        globals()[name].to_parquet(out/f'{name}.parquet',index=False)
    (out/'settings.json').write_text(json.dumps(dict(dominance_factor=DOMINANCE_FACTOR,
        min_plan_depth_m=MIN_PLAN_DEPTH_M,split_depth_m=SPLIT_DEPTH_M,cohort_unit='eddy-day',
        direction='surface-to-depth, geographic bearing clockwise from north',grid_angle_rad=float(grid.angle),
        bootstraps=BOOTSTRAPS,seed=SEED,velocity_composites=RUN_VELOCITY_COMPOSITES,
        target_depths_m=TARGET_DEPTHS_M,matched_depths_m=MATCH_DEPTHS_M),indent=2))
    print(out)
